In [44]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
import numpy as np
import pandas as pd

from src.preprocessing import leave_k_last

In [45]:
input_dir = Path("../data/processed/")

train = pd.read_csv(input_dir / "train.csv")
test = pd.read_csv(input_dir / "test.csv")

In [46]:
unique_users = train["userId"].unique()
unique_items = train["movieId"].unique()
n_users = len(unique_users)
n_items = len(unique_items)

k = 100
mu = train["rating"].mean()
bias_user = np.zeros(n_users)
bias_item = np.zeros(n_items)
p_matrix = np.random.normal(0, 0.01, size=(n_users, k))
q_matrix = np.random.normal(0, 0.01, size=(n_items, k))

user_id_to_idx = {ids: idx for idx, ids in enumerate(unique_users)}
item_id_to_idx = {ids: idx for idx, ids in enumerate(unique_items)}

In [47]:
def rmse(y_true: np.ndarray, y_pred: np.ndarray) ->float:

    return np.sqrt(np.mean((y_true - y_pred)**2))

In [48]:
train_fit, val_fit = leave_k_last(train, k=1)

In [49]:
epochs = 100
learning_rate = 0.01
lm = 0.01

best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0
patience = 10

p_matrix_best = p_matrix.copy()
q_matrix_best = q_matrix.copy()
bias_user_best = bias_user.copy()
bias_item_best = bias_item.copy()

for epoch in range(epochs):

    shuffled_train = train_fit.sample(frac=1, random_state=42)

    train_loss = 0.0
    squared_error = 0.0
    for row in shuffled_train.itertuples():

        u = user_id_to_idx[row.userId]
        i = item_id_to_idx[row.movieId]
        r = row.rating

        rating_pred = mu + bias_user[u] + bias_item[i] + p_matrix[u] @ q_matrix[i]
        error = r - rating_pred

        p_old = p_matrix[u].copy()
        q_old = q_matrix[i].copy()
        p_matrix[u] += learning_rate * (error * q_old - lm * p_old)
        q_matrix[i] += learning_rate * (error * p_old - lm * q_old)
        bias_user[u] += learning_rate * (error - lm * bias_user[u])
        bias_item[i] += learning_rate * (error - lm * bias_item[i])

        squared_error += error**2

    train_loss = np.sqrt(squared_error/len(shuffled_train))

    squared_error = 0.0
    val_loss = 0.0
    for row in val_fit.itertuples():
        u = user_id_to_idx[row.userId]
        i = item_id_to_idx[row.movieId]
        r = row.rating

        rating_pred = mu + bias_user[u] + bias_item[i] + p_matrix[u] @ q_matrix[i]
        squared_error += (r - rating_pred)**2

    val_loss = np.sqrt(squared_error/len(val_fit))

    print(f"[{epoch}/{epochs}] Train loss: {train_loss:.4f}, Val loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        p_matrix_best = p_matrix.copy()
        q_matrix_best = q_matrix.copy()
        bias_user_best = bias_user.copy()
        bias_item_best = bias_item.copy()

        patience_counter = 0

    else:
        patience_counter += 1

        if patience_counter == patience:
            break

[0/100] Train loss: 0.9355, Val loss: 0.9316
[1/100] Train loss: 0.8873, Val loss: 0.9032
[2/100] Train loss: 0.8709, Val loss: 0.8917
[3/100] Train loss: 0.8606, Val loss: 0.8861
[4/100] Train loss: 0.8531, Val loss: 0.8831
[5/100] Train loss: 0.8472, Val loss: 0.8816
[6/100] Train loss: 0.8421, Val loss: 0.8808
[7/100] Train loss: 0.8373, Val loss: 0.8805
[8/100] Train loss: 0.8324, Val loss: 0.8805
[9/100] Train loss: 0.8269, Val loss: 0.8805
[10/100] Train loss: 0.8200, Val loss: 0.8804
[11/100] Train loss: 0.8111, Val loss: 0.8802
[12/100] Train loss: 0.7993, Val loss: 0.8797
[13/100] Train loss: 0.7844, Val loss: 0.8789
[14/100] Train loss: 0.7665, Val loss: 0.8776
[15/100] Train loss: 0.7459, Val loss: 0.8759
[16/100] Train loss: 0.7233, Val loss: 0.8740
[17/100] Train loss: 0.6995, Val loss: 0.8719
[18/100] Train loss: 0.6751, Val loss: 0.8699
[19/100] Train loss: 0.6507, Val loss: 0.8682
[20/100] Train loss: 0.6265, Val loss: 0.8668
[21/100] Train loss: 0.6027, Val loss: 0.865

In [ ]:
p_matrix = p_matrix_best
q_matrix = q_matrix_best
bias_user = bias_user_best
bias_item = bias_item_best